In [ ]:
%pip install azure-identity azure-mgmt-costmanagement
dbutils.library.restartPython()

In [ ]:
%run ./utils_common

In [ ]:
import time
import json
import requests
from azure.core.exceptions import HttpResponseError
from azure.identity import ClientSecretCredential
from azure.mgmt.costmanagement import CostManagementClient
from azure.mgmt.costmanagement.models import (
    QueryDefinition,
    QueryDataset,
    QueryTimePeriod,
    QueryAggregation,
    QueryGrouping,
    ExportType,
    TimeframeType,
)

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("overlap_days", "3")
dbutils.widgets.text("subscription_id", "")
dbutils.widgets.text("scope", "")

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

In [ ]:
azure_logger = setup_logger("AzureCostExplorer")
logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)
logging.getLogger("py4j").setLevel(logging.ERROR)

overlap_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=azure_logger)

In [ ]:
# Table FQNs are constructed inside AzureCostReporterApp for consistency
# with the dependency-injection pattern used across all notebooks.

In [ ]:
class AzureCostClient:
    # Pool VM cost path (plan §4.2/§4.3). Idle/warm pool capacity is tagged
    # DatabricksInstancePoolId but NOT clusterid, so it is invisible to the
    # cluster explorer. We group by BOTH tags and keep only the clusterid-free
    # slice (the §4.3 netting guard) so pool and
    # cluster cloud cost stay disjoint — no double counting across tabs. Azure
    # caps grouping at 2 items, so spending both slots on the two tags leaves no
    # slot for MeterCategory: Azure pool cost lands as a single cloud_cost bucket
    # (compute/storage/network/other left NULL), matching the AWS pool path.
    POOL_TAG_KEY = "databricksinstancepoolid"
    CLUSTER_TAG_KEY = "clusterid"

    # Tenant-level 429 backoff schedule for the top-level cluster/pool query
    # methods. Azure Cost Management enforces a shared tenant quota that can
    # take 30–60 min to reset once tripped, so we bail out slowly (three
    # attempts, escalating) instead of hammering. Kept separate from the
    # generic non-429 retry policy (max_retries=3, wait=2^attempt) which is
    # still appropriate for transient blips.
    RATE_LIMIT_BACKOFF_SEC = (60, 180, 420)

    def __init__(self, subscription_id, tenant_id, client_id, client_secret, logger=None):
        self.subscription_id = subscription_id
        self.logger = logger or logging.getLogger("AzureCostClient")

        self.credential = ClientSecretCredential(
            tenant_id=tenant_id,
            client_id=client_id,
            client_secret=client_secret
        )

        self.client = CostManagementClient(self.credential)
        self.scope = f"/subscriptions/{self.subscription_id}"
        self.max_chunk_days = 10
        self.max_retries = 3
        self.max_rate_limit_retries = len(self.RATE_LIMIT_BACKOFF_SEC)
        # Bounded retries for nextLink pagination so we never spin forever on 429s.
        # Azure tenant-level rate limits can take >1h to reset; better to fail the
        # run and record FAILED than silently hammer the API.
        self.max_page_retries = 8
        # Cap the exponential backoff fallback at 15 minutes per wait.
        self.max_backoff_sec = 900

    # -------- Public API --------
    def build_chunks(self, start_date: datetime, end_date: datetime):
        """Return the native inclusive Azure query chunks for a frozen window."""
        start_utc, end_utc = self._to_utc(start_date, end_date)
        return self._build_chunks(start_utc, end_utc, self.max_chunk_days)

    def query_cluster_chunk(
        self,
        chunk_start: datetime,
        chunk_end: datetime,
        tag_name: str = "clusterid",
    ):
        """Query exactly one chunk; persistence is owned by the reporter app."""
        self.logger.info(f"Querying chunk {chunk_start} → {chunk_end}")
        return self._query_with_retries(chunk_start, chunk_end, tag_name)

    # -------- Helpers: date & chunks --------
    def _to_utc(self, start_date: datetime, end_date: datetime):
        start_utc = start_date.astimezone(timezone.utc)
        end_utc = end_date.astimezone(timezone.utc)
        return start_utc, end_utc

    def _build_chunks(self, start_utc: datetime, end_utc: datetime, max_days: int):
        """Return inclusive (chunk_start, chunk_end) pairs in UTC."""
        chunks = []
        current = start_utc
        while current <= end_utc:
            chunk_end = min(current + timedelta(days=max_days - 1), end_utc)
            chunks.append((current, chunk_end))
            current = chunk_end + timedelta(days=1)
        return chunks

    # -------- Helpers: query construction --------
    def _build_dataset(self, tag_name: str):
        return QueryDataset(
            granularity="Daily",
            aggregation={"totalCost": QueryAggregation(name="Cost", function="Sum")},
            grouping=[
                QueryGrouping(type="TagKey", name=tag_name),
                QueryGrouping(type="Dimension", name="MeterCategory"),
            ],
        )

    def _build_query_definition(self, start_utc: datetime, end_utc: datetime, dataset):
        # AmortizedCost (not ActualCost): this subscription has broad reservation /
        # savings-plan coverage on VMs, so ActualCost bills reservation-covered
        # VM usage at $0 at the resource line (the money sits on a separate
        # untagged reservation-purchase line). Disks + networking are not
        # reservation-eligible so they bill normally regardless — using
        # ActualCost silently zeroes out per-cluster compute_cost while
        # storage/network look fine. Amortized spreads the reservation
        # commitment back across the actual VM-hours, matching AWS behaviour.
        return QueryDefinition(
            type=ExportType.AMORTIZED_COST,
            timeframe=TimeframeType.CUSTOM,
            time_period=QueryTimePeriod(from_property=start_utc, to=end_utc),
            dataset=dataset,
        )

    def _build_query_body_json(self, start_utc: datetime, end_utc: datetime, tag_name: str):
        body = {
            "type": "AmortizedCost",
            "timeframe": "Custom",
            "timePeriod": {
                "from": start_utc.isoformat(),
                "to": end_utc.isoformat(),
            },
            "dataset": {
                "granularity": "Daily",
                "aggregation": {
                    "totalCost": {
                        "name": "Cost",
                        "function": "Sum",
                    }
                },
                "grouping": [
                    {"type": "TagKey", "name": tag_name},
                    {"type": "Dimension", "name": "MeterCategory"},
                ],
            },
        }
        return json.dumps(body)

    # -------- Core call with retries --------
    @staticmethod
    def _is_rate_limited(e):
        """True iff the Azure SDK raised a 429 (tenant/entity/qpu quota).

        Uses HttpResponseError.status_code when available; falls back to a
        string match so non-SDK error wrappers (e.g. re-raised RuntimeError
        from _fetch_next_page) still trigger the long backoff.
        """
        status = getattr(e, "status_code", None)
        if status == 429:
            return True
        text = str(e)
        return "429" in text or "too many requests" in text.lower()

    def _query_with_retries(self, start_utc, end_utc, tag_name):
        dataset = self._build_dataset(tag_name)
        query = self._build_query_definition(start_utc, end_utc, dataset)
        query_json = self._build_query_body_json(start_utc, end_utc, tag_name)

        # Track 429s and other errors on separate counters. Rate-limit backoff
        # (60/180/420s) is meant to survive a tenant-level throttle, while the
        # generic 2^attempt backoff is for transient blips — mixing them would
        # either hammer the API on a real quota trip or waste 8+ minutes on a
        # single network glitch.
        rate_limit_attempt = 0
        error_attempt = 0
        last_exception = None

        while True:
            try:
                return self._execute_query(query, query_json)
            except Exception as e:
                last_exception = e

                if self._is_rate_limited(e):
                    if rate_limit_attempt >= self.max_rate_limit_retries:
                        self.logger.error(
                            f"Rate limited (429) on main query, exhausted "
                            f"{self.max_rate_limit_retries} backoff attempts."
                        )
                        break
                    wait_sec = self.RATE_LIMIT_BACKOFF_SEC[rate_limit_attempt]
                    rate_limit_attempt += 1
                    self.logger.warning(
                        f"Rate limited (429) on main query, waiting {wait_sec}s "
                        f"(rate-limit attempt {rate_limit_attempt}/"
                        f"{self.max_rate_limit_retries})..."
                    )
                    time.sleep(wait_sec)
                else:
                    error_attempt += 1
                    if error_attempt >= self.max_retries:
                        break
                    wait_sec = 2 ** error_attempt
                    self.logger.warning(
                        f"Error on main query, waiting {wait_sec}s "
                        f"(attempt {error_attempt}/{self.max_retries}): {e}"
                    )
                    time.sleep(wait_sec)

        raise last_exception

    # -------- Single query + pagination --------
    def _execute_query(self, query, query_json: str):
        result = self.client.query.usage(self.scope, parameters=query)

        if not result.rows:
            return None

        col_names = None
        if result.columns:
            col_names = [col.name for col in result.columns]

        all_rows = list(result.rows)

        next_link = getattr(result, "next_link", None)
        token = None
        if next_link:
            token = self.credential.get_token(
                "https://management.azure.com/.default"
            ).token

        while next_link:
            next_link, page_rows = self._fetch_next_page(next_link, token, query_json)
            all_rows.extend(page_rows)
            if next_link:
                time.sleep(2)

        return self._rows_to_df(all_rows, col_names)

    def _fetch_next_page(self, next_link: str, token: str, query_json: str):
        attempt = 0
        while True:
            resp = requests.post(
                next_link,
                headers={
                    "Authorization": f"Bearer {token}",
                    "Content-Type": "application/json",
                },
                data=query_json,
            )

            if resp.status_code == 429:
                attempt += 1
                headers = resp.headers

                # Log the full set of rate-limit headers exactly once per page so
                # we can see which scope (qpu/entity/tenant/client) is throttling
                # without flooding logs on every retry.
                if attempt == 1:
                    rate_limit_headers = {
                        k: v for k, v in headers.items()
                        if k.lower().startswith("x-ms-ratelimit") or k.lower() == "retry-after"
                    }
                    self.logger.warning(
                        f"429 on nextLink. Rate-limit headers from Azure: {rate_limit_headers}"
                    )

                if attempt > self.max_page_retries:
                    raise RuntimeError(
                        f"Azure Cost API: pagination still 429 after "
                        f"{self.max_page_retries} retries. Tenant rate limit "
                        f"likely exhausted; aborting run."
                    )

                retry_after = (
                    headers.get("x-ms-ratelimit-microsoft.costmanagement-qpu-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-entity-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-tenant-retry-after")
                    or headers.get("x-ms-ratelimit-microsoft.costmanagement-client-retry-after")
                    or headers.get("Retry-After")
                )

                try:
                    wait_sec = int(retry_after) if retry_after is not None else None
                except (TypeError, ValueError):
                    # Retry-After can occasionally be an HTTP-date; fall back to backoff.
                    wait_sec = None

                if wait_sec is None:
                    # Exponential backoff: 60s, 120s, 240s, 480s, ... capped at max_backoff_sec.
                    # Defaulting to 30s (the previous behavior) is too aggressive for
                    # tenant-level Cost Management throttling.
                    wait_sec = min(60 * (2 ** (attempt - 1)), self.max_backoff_sec)

                self.logger.warning(
                    f"429 throttled, waiting {wait_sec}s before retrying nextLink "
                    f"(attempt {attempt}/{self.max_page_retries})..."
                )
                time.sleep(wait_sec)
                continue

            resp.raise_for_status()
            data = resp.json()
            props = data.get("properties", {})
            page_rows = props.get("rows", [])
            new_next_link = props.get("nextLink")
            return new_next_link, page_rows

    # -------- Helper: convert rows to DataFrame --------
    def _rows_to_df(self, rows, col_names=None):
        """Build a Spark DataFrame from Azure response rows.

        Requires API-reported column names for deterministic schema mapping.
        Raises SchemaValidationError if column metadata is missing or inconsistent.
        """
        if not col_names or not rows or len(col_names) != len(rows[0]):
            raise SchemaValidationError(
                f"Azure Cost API returned rows with {len(rows[0]) if rows else 0} columns "
                f"but reported {len(col_names) if col_names else 0} column names. "
                f"Cannot construct DataFrame without reliable column metadata."
            )
        df = spark.createDataFrame(rows, col_names)
        for c in df.columns:
            df = df.withColumnRenamed(c, c.lower())
        return df

    # =======================================================
    # Pool VM cost path (plan §4.2 / §4.3 / §4.6)
    # =======================================================
    # Grouping by TWO TagKeys (pool + cluster) drives the §4.3 netting guard.
    # This path deliberately BYPASSES _rows_to_df: two TagKey groupings return
    # two ("TagKey", "TagValue") column pairs whose names collide once
    # lowercased, so we parse the raw rows + ordered column names into clean
    # records first (§4.2.1) before building the DataFrame.

    def query_pool_chunk(self, chunk_start: datetime, chunk_end: datetime):
        """Query and parse exactly one pool chunk without cross-chunk buffering."""
        self.logger.info(f"Querying POOL chunk {chunk_start} → {chunk_end}")
        records = self._query_pool_with_retries(chunk_start, chunk_end)
        if not records:
            return None
        return self._pool_records_to_df(records)

    # -------- Pool query construction --------
    def _build_pool_dataset(self):
        return QueryDataset(
            granularity="Daily",
            aggregation={"totalCost": QueryAggregation(name="Cost", function="Sum")},
            grouping=[
                QueryGrouping(type="TagKey", name=self.POOL_TAG_KEY),
                QueryGrouping(type="TagKey", name=self.CLUSTER_TAG_KEY),
            ],
        )

    def _build_pool_query_definition(self, start_utc: datetime, end_utc: datetime, dataset):
        # AmortizedCost — same reasoning as the cluster query (see
        # _build_query_definition). Must stay in sync with the cluster path so
        # pool + cluster VM cost use identical accounting; otherwise the two
        # tabs will disagree during reservation-heavy months.
        return QueryDefinition(
            type=ExportType.AMORTIZED_COST,
            timeframe=TimeframeType.CUSTOM,
            time_period=QueryTimePeriod(from_property=start_utc, to=end_utc),
            dataset=dataset,
        )

    def _build_pool_query_body_json(self, start_utc: datetime, end_utc: datetime):
        body = {
            "type": "AmortizedCost",
            "timeframe": "Custom",
            "timePeriod": {
                "from": start_utc.isoformat(),
                "to": end_utc.isoformat(),
            },
            "dataset": {
                "granularity": "Daily",
                "aggregation": {
                    "totalCost": {
                        "name": "Cost",
                        "function": "Sum",
                    }
                },
                "grouping": [
                    {"type": "TagKey", "name": self.POOL_TAG_KEY},
                    {"type": "TagKey", "name": self.CLUSTER_TAG_KEY},
                ],
            },
        }
        return json.dumps(body)

    # -------- Pool query with retries --------
    def _query_pool_with_retries(self, start_utc, end_utc):
        dataset = self._build_pool_dataset()
        query = self._build_pool_query_definition(start_utc, end_utc, dataset)
        query_json = self._build_pool_query_body_json(start_utc, end_utc)

        # Two independent attempt counters (see _query_with_retries for the
        # split rationale).
        rate_limit_attempt = 0
        error_attempt = 0
        last_exception = None

        while True:
            try:
                rows, col_names = self._execute_pool_query(query, query_json)
                return self._parse_pool_rows(rows, col_names)
            except Exception as e:
                last_exception = e

                if self._is_rate_limited(e):
                    if rate_limit_attempt >= self.max_rate_limit_retries:
                        self.logger.error(
                            f"Rate limited (429) on pool query, exhausted "
                            f"{self.max_rate_limit_retries} backoff attempts."
                        )
                        break
                    wait_sec = self.RATE_LIMIT_BACKOFF_SEC[rate_limit_attempt]
                    rate_limit_attempt += 1
                    self.logger.warning(
                        f"Rate limited (429) on pool query, waiting {wait_sec}s "
                        f"(rate-limit attempt {rate_limit_attempt}/"
                        f"{self.max_rate_limit_retries})..."
                    )
                    time.sleep(wait_sec)
                else:
                    error_attempt += 1
                    if error_attempt >= self.max_retries:
                        break
                    wait_sec = 2 ** error_attempt
                    self.logger.warning(
                        f"Error on pool query, waiting {wait_sec}s "
                        f"(attempt {error_attempt}/{self.max_retries}): {e}"
                    )
                    time.sleep(wait_sec)

        raise last_exception

    # -------- Single pool query + pagination (raw rows) --------
    def _execute_pool_query(self, query, query_json: str):
        """Run the pool query + follow pagination, returning RAW rows and the
        ordered column names (no lowercasing / no DataFrame build).

        The §4.2.1 parser needs the ordered, un-deduplicated columns so it can
        pair the duplicate ("TagKey", "TagValue") columns positionally — which
        _rows_to_df cannot do because it lowercases and would collide the two
        tagvalue columns.
        """
        result = self.client.query.usage(self.scope, parameters=query)

        if not result.rows:
            return [], None

        col_names = None
        if result.columns:
            col_names = [col.name for col in result.columns]

        all_rows = list(result.rows)

        next_link = getattr(result, "next_link", None)
        token = None
        if next_link:
            token = self.credential.get_token(
                "https://management.azure.com/.default"
            ).token

        while next_link:
            next_link, page_rows = self._fetch_next_page(next_link, token, query_json)
            all_rows.extend(page_rows)
            if next_link:
                time.sleep(2)

        return all_rows, col_names

    # -------- Pool response parsing (§4.2.1 + §4.3 netting) --------
    @staticmethod
    def _first_col_index(lower_cols, candidates):
        for cand in candidates:
            if cand in lower_cols:
                return lower_cols.index(cand)
        return None

    @staticmethod
    def _safe_float(value):
        try:
            return float(value)
        except (TypeError, ValueError):
            return 0.0

    def _parse_pool_rows(self, rows, col_names):
        """Parse raw pool rows into netted records (plan §4.2.1).

        Robust path: pair the TagKey/TagValue columns positionally and read each
        pair's TagKey VALUE to learn which tag it is, so the pool↔cluster mapping
        is correct regardless of grouping order or duplicate column names.

        Fallbacks (defensive, logged): if the API surfaces no TagKey columns
        (only two TagValues), map by grouping order (pool first); if only one tag
        value surfaces, treat it as the pool and skip netting. Because the whole
        run_pool path is isolated, a wrong assumption fails SAFE — it logs the
        actual columns and leaves the cluster explorer untouched.
        """
        if not rows:
            return []
        if not col_names:
            self.logger.warning(
                "Pool query returned rows but no column metadata; skipping parse."
            )
            return []

        lower_cols = [c.lower() for c in col_names]
        cost_idx = self._first_col_index(lower_cols, ("cost", "pretaxcost", "costusd"))
        currency_idx = self._first_col_index(lower_cols, ("currency", "billingcurrency"))
        date_idx = self._first_col_index(lower_cols, ("usagedate", "date"))

        if cost_idx is None or date_idx is None:
            self.logger.warning(
                f"Pool parse cannot locate cost/date columns; skipping. "
                f"API column names: {col_names}"
            )
            return []

        # Pair TagKey/TagValue columns positionally (a TagKey column is followed
        # by its TagValue column).
        tag_pairs = []
        i, n = 0, len(lower_cols)
        while i < n:
            if lower_cols[i] == "tagkey":
                val_idx = i + 1 if (i + 1 < n and lower_cols[i + 1] == "tagvalue") else None
                tag_pairs.append((i, val_idx))
                i += 2 if val_idx is not None else 1
            else:
                i += 1
        tag_value_idxs = [idx for idx, c in enumerate(lower_cols) if c == "tagvalue"]

        keyed = bool(tag_pairs) and all(v is not None for _, v in tag_pairs)
        fallback = None

        records = []
        for row in rows:
            cost = self._safe_float(row[cost_idx])
            if cost == 0.0:
                continue
            currency = row[currency_idx] if currency_idx is not None else "USD"
            raw_date = row[date_idx]

            netting_applied = True
            if keyed:
                tags = {}
                for k_idx, v_idx in tag_pairs:
                    key = (str(row[k_idx]) if row[k_idx] is not None else "").strip().lower()
                    tags[key] = row[v_idx]
                pool_id = tags.get(self.POOL_TAG_KEY)
                cluster_id = tags.get(self.CLUSTER_TAG_KEY)
            elif len(tag_value_idxs) >= 2:
                fallback = "tagvalue-order (pool-first)"
                pool_id = row[tag_value_idxs[0]]
                cluster_id = row[tag_value_idxs[1]]
            elif len(tag_value_idxs) == 1:
                fallback = "single-tagvalue (netting skipped)"
                pool_id = row[tag_value_idxs[0]]
                cluster_id = None
                netting_applied = False
            else:
                fallback = "no-tag-columns"
                continue

            # Netting guard (§4.3): drop pool cost that ALSO carries a clusterid —
            # that slice already lives in dbspend360_cloud_cost_explorer, so
            # keeping it here would double-count across the Pools and Job/
            # All-Purpose tabs.
            if netting_applied and cluster_id is not None and str(cluster_id).strip():
                continue
            if pool_id is None or not str(pool_id).strip():
                continue

            records.append({
                "instance_pool_id": str(pool_id).strip(),
                "cost": cost,
                "currency": str(currency) if currency else "USD",
                "cost_incurred_date": str(raw_date) if raw_date is not None else None,
            })

        if fallback:
            self.logger.warning(
                f"Pool parse used fallback '{fallback}'. API column names: {col_names}"
            )

        return records

    def _pool_records_to_df(self, records):
        """Convert netted pool records to a Spark DataFrame with date typing.

        Azure UsageDate arrives as an int like 20260601 (yyyyMMdd), matching the
        cluster path's date handling.
        """
        df = spark.createDataFrame(records)
        return df.withColumn(
            "cost_incurred_date",
            F.to_date(F.col("cost_incurred_date").cast("string"), "yyyyMMdd"),
        )


In [ ]:
# =======================================================
# Azure MeterCategory Classification Framework
# =======================================================
# Two-stage classifier: exact-match first, then case-insensitive substring.
#
# Exact-match layer (AZURE_METER_EXACT_CLASSIFICATION) exists so the "Virtual
# Machines" MeterCategory (real VM hours -> compute) stays disjoint from
# "Virtual Machines Licenses" / "Virtual Machine Licenses" (BYOL surcharge
# meters that carry no compute time and must land in other_cost). A naive
# `"virtual machine" in "virtual machines licenses"` substring match folds the
# licensing surcharge into compute_cost and inflates the VM number, while
# `"virtual machine" in "virtual network"` catches network meters before the
# network rule gets a chance.
#
# Substring layer (AZURE_METER_CLASSIFICATION) handles the long tail of
# storage / networking meters where substring matching is safe (there is no
# "Storage Licenses" meter, "Bandwidth" only means bandwidth, etc.).
#
# Anything not matching either layer -> "other" (never silently -> compute).
AZURE_METER_EXACT_CLASSIFICATION = {
    "compute": {"virtual machines"},
    "other": {"virtual machines licenses", "virtual machine licenses"},
}

AZURE_METER_CLASSIFICATION = {
    "storage": ["storage", "disk"],
    "network": ["bandwidth", "virtual network", "load balancer", "network watcher"],
}


def _azure_exact_lookup():
    """Flatten AZURE_METER_EXACT_CLASSIFICATION into meter_lower -> category."""
    out = {}
    for category, meters in AZURE_METER_EXACT_CLASSIFICATION.items():
        for m in meters:
            out[m.lower()] = category
    return out


_AZURE_EXACT_LOOKUP = _azure_exact_lookup()


def classify_azure_meter_category(meter_category: str) -> str:
    """Classify an Azure MeterCategory string.

    Order:
      1. Exact (case-insensitive) match against AZURE_METER_EXACT_CLASSIFICATION.
      2. Substring match against AZURE_METER_CLASSIFICATION.
      3. Fallback -> 'other' (unknown categories are logged upstream).

    Unknown categories return 'other' -- never silently 'compute'.
    """
    category, _ = _classify_azure_meter_with_source(meter_category)
    return category


def _classify_azure_meter_with_source(meter_category: str):
    """Return (category, matched_rule) for a meter name.

    matched_rule is True when the meter matched an exact-map or substring rule
    (including a known-other rule like the licensing surcharges), False when
    the meter fell through to the 'other' fallback (i.e. genuinely
    unclassified). Used by _log_unclassified_meters so known-other meters
    aren't reported as unknown every run.
    """
    if not meter_category:
        return "other", False
    lower = meter_category.lower().strip()
    exact = _AZURE_EXACT_LOOKUP.get(lower)
    if exact is not None:
        return exact, True
    for category, patterns in AZURE_METER_CLASSIFICATION.items():
        for pattern in patterns:
            if pattern in lower:
                return category, True
    return "other", False


def build_azure_category_column(mc_col_name: str):
    """Build a PySpark Column expression matching classify_azure_meter_category.

    Exact-match layer first (case-insensitive, whitespace-trimmed), then the
    substring layer. Unknown meters map to 'other'. Kept in lockstep with the
    Python classifier so the Spark path and the unclassified-meter logger
    can never disagree.
    """
    normalized = F.lower(F.trim(F.col(mc_col_name)))

    expr = None
    for category, meters in AZURE_METER_EXACT_CLASSIFICATION.items():
        meter_list = [m.lower() for m in meters]
        cond = normalized.isin(meter_list)
        if expr is None:
            expr = F.when(cond, F.lit(category))
        else:
            expr = expr.when(cond, F.lit(category))

    for category, patterns in AZURE_METER_CLASSIFICATION.items():
        cond = normalized.contains(patterns[0])
        for p in patterns[1:]:
            cond = cond | normalized.contains(p)
        if expr is None:
            expr = F.when(cond, F.lit(category))
        else:
            expr = expr.when(cond, F.lit(category))

    return expr.otherwise(F.lit("other"))


# Known column names the Azure Cost API may use for the tag-value field
_AZURE_TAG_VALUE_CANDIDATES = {"clusterid", "clusteridvalue", "tagvalue"}


def _resolve_cluster_id_column(spark_df, date_col, logger):
    """Rename the Azure tag-value column to ``cluster_id`` if needed.

    Uses only the known candidate column names from _AZURE_TAG_VALUE_CANDIDATES.
    Raises SchemaValidationError if no candidate matches.
    """
    if "cluster_id" in spark_df.columns:
        return spark_df

    tag_value_col = next(
        (c for c in spark_df.columns if c.lower() in _AZURE_TAG_VALUE_CANDIDATES),
        None,
    )

    if tag_value_col is None:
        raise SchemaValidationError(
            f"Cannot resolve cluster_id column from API response. "
            f"Columns present: {spark_df.columns}. "
            f"Expected one of: {sorted(_AZURE_TAG_VALUE_CANDIDATES)}"
        )

    logger.info(f"Resolved cluster_id column from '{tag_value_col}'")
    return spark_df.withColumnRenamed(tag_value_col, "cluster_id")


# =======================================================
# APP
# =======================================================
class AzureCostReporterApp:
    """Orchestrates incremental Azure cost ingestion into the cloud cost table.

    Invariant enforced: cloud_cost = compute_cost + storage_cost + network_cost + other_cost
    """

    TABLE_NAME = "dbspend360_cloud_cost_explorer"
    # Pool VM explorer target (plan §4.2). Separate audit watermark + table so
    # the pool path backfills and MERGEs independently of the cluster path.
    POOL_TABLE_NAME = "dbspend360_pool_cloud_cost_explorer"

    # Post-write monitor (plan §4.6). Window pool cloud_cost below this floor
    # while pool DBU is non-zero ⇒ suspected DatabricksInstancePoolId tagging
    # lapse; a non-silent alarm fires (advisory, never fails the run).
    CLOUD_COST_FLOOR = 0.01

    def __init__(self, catalog, schema, overlap_days, logger):
        self.overlap_days = overlap_days
        self.logger = logger
        self.audit_table = build_table_fqn(catalog, schema, "dbspend360_audit_log")
        self.target_table = build_table_fqn(catalog, schema, "dbspend360_cloud_cost_explorer")
        self.pool_target_table = build_table_fqn(catalog, schema, self.POOL_TABLE_NAME)
        # Pool DBU table — read-only, best-effort cross-check for the pool
        # monitor (§4.6): only alarm on ~0 pool cloud when pool DBU exists.
        self.pool_dbu_table = build_table_fqn(catalog, schema, "dbspend360_pool_dbu_cost")
        self.error_log_table = build_table_fqn(catalog, schema, "dbspend360_error_log")
        self.breakdown_table = build_table_fqn(catalog, schema, "dbspend360_other_cost_breakdown")

        scope = dbutils.widgets.get("scope")
        subscription_id = dbutils.widgets.get("subscription_id")
        tenant_id = dbutils.secrets.get(scope, "tenant_id")
        client_id = dbutils.secrets.get(scope, "client_id")
        client_secret = dbutils.secrets.get(scope, "client_secret")

        self.client = AzureCostClient(
            subscription_id, tenant_id, client_id, client_secret,
            logger=self.logger,
        )

    @staticmethod
    def _chunk_checkpoint_name(table_name, chunk_start, chunk_end):
        """Build a deterministic key disjoint from the standard watermark key."""
        return (
            f"{table_name}__chunk__{chunk_start:%Y-%m-%d}"
            f"__{chunk_end:%Y-%m-%d}"
        )

    def _successful_chunk_row_count(self, checkpoint_name):
        """Return the latest durable SUCCESS row count, or None if incomplete."""
        rows = (
            spark.table(self.audit_table)
            .filter(
                (F.col("table_name") == F.lit(checkpoint_name))
                & (F.col("status") == F.lit("SUCCESS"))
            )
            .orderBy(F.col("created_at").desc())
            .select("row_count")
            .limit(1)
            .collect()
        )
        return int(rows[0][0] or 0) if rows else None

    def _record_chunk_failure(
        self, table_name, chunk_start, chunk_end, error,
    ):
        checkpoint_name = self._chunk_checkpoint_name(
            table_name, chunk_start, chunk_end,
        )
        try:
            log_audit_run(
                self.audit_table, checkpoint_name, chunk_start, chunk_end,
                "FAILED", 0, str(error)[:1000],
            )
        except Exception:
            self.logger.error(
                f"Failed to write FAILED chunk checkpoint {checkpoint_name}"
            )

    def _merge_cluster_chunk(self, spark_df, chunk_start, chunk_end):
        """Transform, validate, and merge one cluster chunk."""
        if spark_df is None or spark_df.limit(1).count() == 0:
            raise DataQualityError(
                "Azure returned an empty cluster-cost response for "
                f"{chunk_start} → {chunk_end}; refusing to checkpoint it as success."
            )
        else:
            date_col = (
                "usagedate"
                if "usagedate" in [c.lower() for c in spark_df.columns]
                else "date_key"
            )
            spark_df = spark_df.withColumn(
                "cost_incurred_date",
                F.to_date(F.col(date_col).cast("string"), "yyyyMMdd"),
            )
            spark_df = _resolve_cluster_id_column(spark_df, date_col, self.logger)
            inc_df = filter_valid_cost_rows(spark_df)

            if inc_df.limit(1).count() == 0:
                self.logger.info(
                    f"No valid cluster rows for chunk {chunk_start} → {chunk_end}."
                )
                merged_row_count = 0
                quality_msg = f"overlap_days={self.overlap_days}, filtered_empty=true"
            else:
                meter_candidates = {"metercategory", "meter_category"}
                mc_col = next(
                    (c for c in inc_df.columns if c.lower() in meter_candidates),
                    None,
                )
                if mc_col is not None:
                    self._log_unclassified_meters(inc_df, mc_col)
                    classified = inc_df.withColumn(
                        "category", build_azure_category_column(mc_col),
                    )
                    write_other_cost_breakdown(
                        classified, mc_col, "AZURE", self.breakdown_table,
                        logger=self.logger,
                    )
                    agg_df = aggregate_costs_by_category(classified)
                else:
                    self.logger.warning(
                        "MeterCategory column not found; all cost assigned to cloud_cost only."
                    )
                    agg_df = (
                        inc_df
                        .groupBy("cluster_id", "currency", "cost_incurred_date")
                        .agg(F.sum("cost").alias("cloud_cost"))
                        .withColumn("compute_cost", F.lit(None).cast("double"))
                        .withColumn("storage_cost", F.lit(None).cast("double"))
                        .withColumn("network_cost", F.lit(None).cast("double"))
                        .withColumn("other_cost", F.lit(None).cast("double"))
                        .withColumn("created_at", F.current_timestamp())
                        .withColumn("updated_at", F.current_timestamp())
                    )

                agg_df = safe_cache(agg_df)
                try:
                    merged_row_count = agg_df.count()
                    validate_source_schema(
                        agg_df,
                        {"cluster_id": "string", "currency": "string",
                         "cost_incurred_date": "date", "cloud_cost": "double"},
                        self.target_table, self.logger,
                    )
                    validate_no_negative_costs(
                        agg_df,
                        ["cloud_cost", "compute_cost", "storage_cost",
                         "network_cost", "other_cost"],
                        self.target_table, self.logger,
                    )
                    validate_currency_consistency(
                        agg_df, "currency", self.target_table, self.logger,
                    )
                    quality_msg = compute_quality_metrics(
                        agg_df, merged_row_count, self.overlap_days,
                        logger=self.logger,
                    )
                    merge_cloud_cost_explorer(self.target_table, agg_df)
                finally:
                    safe_unpersist(agg_df)

                merge_metrics = get_merge_metrics(self.target_table, self.logger)
                quality_msg += (
                    f", merge_inserted={merge_metrics.get('num_inserted', '?')}"
                    f", merge_updated={merge_metrics.get('num_updated', '?')}"
                )

        validate_post_merge(
            self.target_table, "cost_incurred_date",
            chunk_start, chunk_end, merged_row_count, self.logger,
        )
        return merged_row_count, quality_msg

    def run(self):
        """Load cluster cost one durable 10-day checkpoint at a time."""
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            # Freeze the table-level bounds once for this attempt. Chunk audit
            # writes must not cause a moving end date or watermark mid-run.
            start_dt, end_dt = get_date_window(
                self.audit_table, self.TABLE_NAME, self.overlap_days,
            )
            self.logger.info(
                f"Querying Azure cost from {start_dt} to {end_dt} "
                f"(overlap_days={self.overlap_days})"
            )
            valid, msg = validate_date_window(start_dt, end_dt, self.overlap_days)
            if not valid:
                raise DataQualityError(msg)

            ensure_cost_columns(self.target_table, logger=self.logger)
            chunks = self.client.build_chunks(
                datetime.combine(start_dt, datetime.min.time(), tzinfo=timezone.utc),
                datetime.combine(end_dt, datetime.max.time(), tzinfo=timezone.utc),
            )
            merged_row_count = 0
            completed_chunks = 0

            for chunk_start_utc, chunk_end_utc in chunks:
                chunk_start = chunk_start_utc.date()
                chunk_end = chunk_end_utc.date()
                checkpoint_name = self._chunk_checkpoint_name(
                    self.TABLE_NAME, chunk_start, chunk_end,
                )
                checkpoint_rows = self._successful_chunk_row_count(checkpoint_name)
                if checkpoint_rows is not None:
                    self.logger.info(
                        f"Skipping durable cluster checkpoint {checkpoint_name}."
                    )
                    merged_row_count += checkpoint_rows
                    completed_chunks += 1
                    continue

                try:
                    spark_df = self.client.query_cluster_chunk(
                        chunk_start_utc, chunk_end_utc, tag_name="clusterid",
                    )
                    chunk_rows, chunk_quality = self._merge_cluster_chunk(
                        spark_df, chunk_start, chunk_end,
                    )
                    # SUCCESS is intentionally after MERGE and post-merge
                    # validation. A crash before this append safely replays the
                    # idempotent natural-key MERGE on the next attempt.
                    log_audit_run(
                        self.audit_table, checkpoint_name, chunk_start, chunk_end,
                        "SUCCESS", chunk_rows, chunk_quality,
                    )
                except Exception as chunk_error:
                    self._record_chunk_failure(
                        self.TABLE_NAME, chunk_start, chunk_end, chunk_error,
                    )
                    raise

                merged_row_count += chunk_rows
                completed_chunks += 1
                time.sleep(5)

            if completed_chunks != len(chunks):
                raise DataQualityError(
                    f"Only {completed_chunks}/{len(chunks)} cluster chunks completed."
                )

            quality_msg = (
                f"overlap_days={self.overlap_days}, "
                f"chunks_completed={completed_chunks}/{len(chunks)}, "
                f"rows={merged_row_count}, checkpointed=true"
            )
            self.logger.info(
                f"Merged {merged_row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt} across {completed_chunks} chunks."
            )
            # This standard key is the only watermark consumed by get_date_window
            # and is withheld until every expected chunk is durable.
            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", merged_row_count, quality_msg,
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

    def _merge_pool_chunk(self, spark_df, chunk_start, chunk_end):
        """Transform, validate, and merge one isolated pool chunk."""
        quality_msg = f"overlap_days={self.overlap_days}, empty_chunk=true"
        if spark_df is None or spark_df.limit(1).count() == 0:
            self.logger.info(
                f"No Azure pool cost data for chunk {chunk_start} → {chunk_end}."
            )
            merged_row_count = 0
        else:
            inc_df = (
                spark_df
                .filter(
                    (F.col("instance_pool_id").isNotNull())
                    & (F.col("instance_pool_id") != "")
                )
                .filter(F.col("cost_incurred_date").isNotNull())
            )
            if inc_df.limit(1).count() == 0:
                self.logger.info(
                    f"No valid pool rows for chunk {chunk_start} → {chunk_end}."
                )
                merged_row_count = 0
                quality_msg = f"overlap_days={self.overlap_days}, filtered_empty=true"
            else:
                agg_df = safe_cache(
                    inc_df
                    .groupBy("instance_pool_id", "currency", "cost_incurred_date")
                    .agg(F.sum("cost").alias("cloud_cost"))
                    .withColumn("compute_cost", F.lit(None).cast("double"))
                    .withColumn("storage_cost", F.lit(None).cast("double"))
                    .withColumn("network_cost", F.lit(None).cast("double"))
                    .withColumn("other_cost", F.lit(None).cast("double"))
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                )
                try:
                    merged_row_count = agg_df.count()
                    validate_source_schema(
                        agg_df,
                        {"instance_pool_id": "string", "currency": "string",
                         "cost_incurred_date": "date", "cloud_cost": "double"},
                        self.pool_target_table, self.logger,
                    )
                    validate_no_negative_costs(
                        agg_df, ["cloud_cost"], self.pool_target_table, self.logger,
                    )
                    validate_currency_consistency(
                        agg_df, "currency", self.pool_target_table, self.logger,
                    )
                    quality_msg = (
                        f"overlap_days={self.overlap_days}, rows={merged_row_count}, "
                        "classification=n/a (single-bucket pool VM), "
                        "netting=clusterid-excluded"
                    )
                    self.logger.info(f"Pool data quality: {quality_msg}")
                    merge_pool_cloud_cost_explorer(self.pool_target_table, agg_df)
                finally:
                    safe_unpersist(agg_df)

                merge_metrics = get_merge_metrics(self.pool_target_table, self.logger)
                quality_msg += (
                    f", merge_inserted={merge_metrics.get('num_inserted', '?')}"
                    f", merge_updated={merge_metrics.get('num_updated', '?')}"
                )

        validate_post_merge(
            self.pool_target_table, "cost_incurred_date",
            chunk_start, chunk_end, merged_row_count, self.logger,
        )
        return merged_row_count, quality_msg

    def _pool_window_cost(self, start_dt, end_dt):
        value = (
            spark.table(self.pool_target_table)
            .filter(
                (F.col("cost_incurred_date") >= F.lit(start_dt))
                & (F.col("cost_incurred_date") <= F.lit(end_dt))
            )
            .agg(F.sum("cloud_cost"))
            .collect()[0][0]
        )
        return float(value or 0.0)

    def run_pool(self):
        """Load pool cost in checkpoints and fail on incomplete ingestion."""
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            # Freeze independent pool bounds once for this attempt.
            start_dt, end_dt = get_date_window(
                self.audit_table, self.POOL_TABLE_NAME, self.overlap_days,
            )
            self.logger.info(
                f"Querying Azure POOL cost from {start_dt} to {end_dt} "
                f"(overlap_days={self.overlap_days})"
            )
            valid, msg = validate_date_window(start_dt, end_dt, self.overlap_days)
            if not valid:
                raise DataQualityError(msg)

            chunks = self.client.build_chunks(
                datetime.combine(start_dt, datetime.min.time(), tzinfo=timezone.utc),
                datetime.combine(end_dt, datetime.max.time(), tzinfo=timezone.utc),
            )
            merged_row_count = 0
            completed_chunks = 0

            for chunk_start_utc, chunk_end_utc in chunks:
                chunk_start = chunk_start_utc.date()
                chunk_end = chunk_end_utc.date()
                checkpoint_name = self._chunk_checkpoint_name(
                    self.POOL_TABLE_NAME, chunk_start, chunk_end,
                )
                checkpoint_rows = self._successful_chunk_row_count(checkpoint_name)
                if checkpoint_rows is not None:
                    self.logger.info(
                        f"Skipping durable pool checkpoint {checkpoint_name}."
                    )
                    merged_row_count += checkpoint_rows
                    completed_chunks += 1
                    continue

                try:
                    spark_df = self.client.query_pool_chunk(
                        chunk_start_utc, chunk_end_utc,
                    )
                    chunk_rows, chunk_quality = self._merge_pool_chunk(
                        spark_df, chunk_start, chunk_end,
                    )
                    log_audit_run(
                        self.audit_table, checkpoint_name, chunk_start, chunk_end,
                        "SUCCESS", chunk_rows, chunk_quality,
                    )
                except Exception as chunk_error:
                    self._record_chunk_failure(
                        self.POOL_TABLE_NAME, chunk_start, chunk_end, chunk_error,
                    )
                    raise

                merged_row_count += chunk_rows
                completed_chunks += 1
                time.sleep(5)

            if completed_chunks != len(chunks):
                raise DataQualityError(
                    f"Only {completed_chunks}/{len(chunks)} pool chunks completed."
                )

            # Read the frozen window from the durable target so resumed/skipped
            # chunks participate in the existing monitor.
            total_pool_cost = self._pool_window_cost(start_dt, end_dt)
            monitor_alerts = self._monitor_pool_post_write(
                total_pool_cost, start_dt, end_dt,
            )
            quality_msg = (
                f"overlap_days={self.overlap_days}, "
                f"chunks_completed={completed_chunks}/{len(chunks)}, "
                f"rows={merged_row_count}, checkpointed=true, "
                f"monitor_alerts={len(monitor_alerts)}"
            )
            self.logger.info(
                f"Merged {merged_row_count} pool rows into {self.pool_target_table} "
                f"for {start_dt} → {end_dt} across {completed_chunks} chunks."
            )
            # Standard pool SUCCESS is withheld until every pool chunk is durable.
            log_audit_run(
                self.audit_table, self.POOL_TABLE_NAME, start_dt, end_dt,
                "SUCCESS", merged_row_count, quality_msg,
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Pool explorer run failed: {msg}")
            try:
                write_error_log_entries(
                    [f"Pool Cost Management explorer failed for {start_dt} → {end_dt}: {msg}"],
                    "AZURE", "POOL_COST_EXPLORER_FAILED", self.error_log_table,
                )
            except Exception:
                self.logger.error("Failed to persist pool explorer failure to error_log")
            try:
                log_audit_run(
                    self.audit_table, self.POOL_TABLE_NAME, start_dt, end_dt,
                    "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED pool audit entry")
            # Pool cloud is part of the tab's headline total. Propagate failure
            # so the job cannot report SUCCESS and refresh downstream rollups
            # from a stale cloud table.
            raise

    def _pool_dbu_present(self, start_dt, end_dt):
        """Best-effort check: does pool DBU exist in the window? (§4.6 monitor).

        Used only to suppress a false ~0-cloud alarm on a genuinely idle/empty
        window. Conservative: returns False (no alarm) if the table can't be
        read, so the monitor never fails the run.
        """
        try:
            cnt = (
                spark.table(self.pool_dbu_table)
                .filter(
                    (F.col("usage_date") >= F.lit(start_dt))
                    & (F.col("usage_date") <= F.lit(end_dt))
                )
                .limit(1)
                .count()
            )
            return cnt > 0
        except Exception as e:
            self.logger.warning(
                f"Could not read pool DBU table for monitor cross-check: {e}"
            )
            return False

    def _monitor_pool_post_write(self, total_pool_cost, start_dt, end_dt):
        """Post-write monitor / alarm for the pool path (plan §4.6).

        If window pool cloud_cost collapses to ~0 while pool DBU is non-zero,
        raise a non-silent alarm (suspected DatabricksInstancePoolId tag lapse
        or empty Cost Management response). A ~0 cloud with no pool DBU is a
        genuinely idle/empty window, not an error, so no alarm fires there.
        Advisory only — never fails the run. Returns the alert list.
        """
        alerts = []

        if total_pool_cost < self.CLOUD_COST_FLOOR:
            if self._pool_dbu_present(start_dt, end_dt):
                alerts.append(
                    f"Azure pool cloud_cost for {start_dt} → {end_dt} collapsed to "
                    f"{total_pool_cost:.4f} (< {self.CLOUD_COST_FLOOR}) while pool "
                    f"DBU is non-zero; suspected DatabricksInstancePoolId tag "
                    f"lapse or empty Cost Management response."
                )
            else:
                self.logger.info(
                    f"Pool cloud_cost ~0 for {start_dt} → {end_dt} but no pool DBU "
                    f"in window; treating as genuinely idle/empty — no alarm."
                )

        if alerts:
            for alert in alerts:
                self.logger.error(f"[POOL MONITOR ALARM] {alert}")
            try:
                write_error_log_entries(
                    alerts, "AZURE", "POOL_COST_MONITOR_ALARM", self.error_log_table,
                )
            except Exception as e:
                self.logger.error(
                    f"Failed to persist pool monitor alarm to error_log: {e}"
                )
        else:
            self.logger.info(
                f"Pool post-write monitor OK for {start_dt} → {end_dt}: "
                f"cloud_cost={total_pool_cost:.4f}."
            )

        return alerts

    def _log_unclassified_meters(self, df, mc_col):
        """Log unclassified MeterCategory values to both logger and error_log table."""
        meter_costs = (
            df.groupBy(mc_col)
            .agg(
                F.sum("cost").alias("total_cost"),
                F.count("*").alias("row_count"),
            )
            .collect()
        )
        unknown_rows = [
            r for r in meter_costs
            if r[0] and not _classify_azure_meter_with_source(r[0])[1]
        ]

        if not unknown_rows:
            return

        meter_names = [r[0] for r in unknown_rows]
        self.logger.warning(
            f"Unclassified Azure MeterCategory values (routed to other_cost): {meter_names}"
        )

        try:
            error_details = [
                f"Unclassified meter: {r[0]}, total_cost=${r.total_cost:.4f}, rows={r.row_count}"
                for r in unknown_rows
            ]
            write_error_log_entries(error_details, "AZURE", "UNCLASSIFIED_COST", self.error_log_table)
        except Exception as e:
            self.logger.warning(f"Failed to write unclassified meters to error_log: {e}")

In [ ]:
# =======================================================
# Execute
# =======================================================
app = AzureCostReporterApp(catalog, schema, overlap_days, azure_logger)
# Both paths are required for a successful task. In particular, pool-cloud
# failures must propagate so downstream pool rollups cannot refresh against a
# stale explorer while the job reports SUCCESS.
app.run()
app.run_pool()